# CIFAR-10 Active & Passive Diffusion — Training + FID

Runs the full pipeline:
1. **Passive** — Phase 1 (epochs 1–1000, `lr=1e-4`, `EMA=0.997`) → Phase 2 (epochs 1001–2600, `lr=5e-5`, `EMA=0.9997`)
2. **Active**  — Phase 1 (epochs 1–1000, `lr=1e-4`, `EMA=0.997`) → Phase 2 (epochs 1001–2600, `lr=5e-5`, `EMA=0.9997`)
3. **Sample generation** for both
4. **FID computation** against CIFAR-10 train stats

Run each cell independently. To resume from a checkpoint, set `--ckpt` and `--epochs` as needed.

## Setup

In [ ]:
import os, torch

# Run from the directory containing train_multigpu.py
%cd /home/agnish/Documents/Projects/Final_Diffusion_Paper_code/CIFAR10

n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}")
for i in range(n_gpus):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")

os.makedirs("results_cifar10_passive", exist_ok=True)
os.makedirs("results_cifar10_active",  exist_ok=True)

---
## Passive Diffusion — Phase 1  (epochs 1–1000)

SDE: `dx = -k·x dt + √(2Tp) dW`  
`Tp=6.4`, `k=4.0`, `T=2.0` · `lr=1e-4` · `EMA decay=0.997`

In [ ]:
!torchrun --standalone --nproc_per_node=4 train_multigpu.py \
  --out_dir results_cifar10_passive \
  --epochs 1000 \
  --batch_size 128 \
  --model_base_dim 128 \
  --num_res_blocks 4 \
  --dim_mults 1,2,2,2 \
  --attn_resolutions 16 \
  --lr 0.0001 \
  --Tp 6.4 \
  --k 4.0 \
  --T 2.0 \
  --model_ema_decay 0.997 \
  --save_freq 200 \
  --large_sample_interval 200 \
  --large_sample_count 200 \
  --pf_sample_interval 200 \
  --pf_sample_count 200 \
  --pf_steps 400 \
  --pf_schedule log \
  --warmup_steps 5000 \
  --timesteps 1000 \
  --grad_clip 1.0 \
  --amp

## Passive Diffusion — Phase 2  (epochs 1001–2600)

Resumed from `checkpoint_epoch_1000.pt`.  
Key changes: **`lr=5e-5`** · **`EMA decay=0.9997`**

In [ ]:
!torchrun --standalone --nproc_per_node=4 train_multigpu.py \
  --ckpt results_cifar10_passive/checkpoint_epoch_1000.pt \
  --out_dir results_cifar10_passive \
  --epochs 2600 \
  --batch_size 128 \
  --model_base_dim 128 \
  --num_res_blocks 4 \
  --dim_mults 1,2,2,2 \
  --attn_resolutions 16 \
  --lr 0.00005 \
  --Tp 6.4 \
  --k 4.0 \
  --T 2.0 \
  --model_ema_decay 0.9997 \
  --save_freq 200 \
  --large_sample_interval 200 \
  --large_sample_count 200 \
  --pf_sample_interval 200 \
  --pf_sample_count 200 \
  --pf_steps 400 \
  --pf_schedule log \
  --warmup_steps 5000 \
  --timesteps 1000 \
  --grad_clip 1.0 \
  --amp

---
## Active Diffusion — Phase 1  (epochs 1–1000)

SDE:
```
dx = (-k·x + η) dt + √(2Tp) dW_x
dη = (-η/τ)    dt + (1/τ)√(2Ta) dW_η
```
`Ta=6.4`, `Tp=1e-3`, `k=4.0`, `τ=0.15` · `lr=1e-4` · `EMA decay=0.997`

In [ ]:
!torchrun --standalone --nproc_per_node=4 train_multigpu.py \
  --active \
  --amp \
  --out_dir results_cifar10_active \
  --epochs 1000 \
  --batch_size 128 \
  --model_base_dim 128 \
  --num_res_blocks 4 \
  --dim_mults 1,2,2,2 \
  --attn_resolutions 16 \
  --lr 0.0001 \
  --Tp 1e-3 \
  --Ta 6.4 \
  --k 4.0 \
  --tau 0.15 \
  --T 2.0 \
  --model_ema_decay 0.997 \
  --save_freq 200 \
  --large_sample_interval 200 \
  --large_sample_count 200 \
  --pf_sample_interval 200 \
  --pf_sample_count 200 \
  --pf_steps 400 \
  --pf_schedule log \
  --warmup_steps 5000 \
  --timesteps 1000 \
  --grad_clip 1.0 \
  --weight_decay 0.0

## Active Diffusion — Phase 2  (epochs 1001–2600)

Resumed from `checkpoint_epoch_1000.pt`.  
Key changes: **`lr=5e-5`** · **`EMA decay=0.9997`**

In [ ]:
!torchrun --standalone --nproc_per_node=4 train_multigpu.py \
  --active \
  --amp \
  --ckpt results_cifar10_active/checkpoint_epoch_1000.pt \
  --out_dir results_cifar10_active \
  --epochs 2600 \
  --batch_size 128 \
  --model_base_dim 128 \
  --num_res_blocks 4 \
  --dim_mults 1,2,2,2 \
  --attn_resolutions 16 \
  --lr 0.00005 \
  --Tp 1e-3 \
  --Ta 6.4 \
  --k 4.0 \
  --tau 0.15 \
  --T 2.0 \
  --model_ema_decay 0.9997 \
  --save_freq 200 \
  --large_sample_interval 200 \
  --large_sample_count 200 \
  --pf_sample_interval 200 \
  --pf_sample_count 200 \
  --pf_steps 400 \
  --pf_schedule log \
  --warmup_steps 5000 \
  --timesteps 1000 \
  --grad_clip 1.0 \
  --weight_decay 0.0

---
## Sample Generation — Passive

40 000 images via the probability-flow ODE sampler.

In [ ]:
!torchrun --standalone --nproc_per_node=4 generate_samples_multigpu.py \
  --ckpt results_cifar10_passive/checkpoint_epoch_2600.pt \
  --model_base_dim 128 \
  --num_res_blocks 4 \
  --attn_resolutions 16 \
  --num_samples 40000 \
  --batch_size 512 \
  --output_dir results_cifar10_passive/fid_samples_2600 \
  --grid_out results_cifar10_passive/sample_grid_2600.png \
  --Tp 6.4 \
  --k 4.0 \
  --T 2.0 \
  --probability_flow \
  --pf_steps 600 \
  --pf_schedule log

## Sample Generation — Active

In [ ]:
!torchrun --standalone --nproc_per_node=4 generate_samples_multigpu.py \
  --ckpt results_cifar10_active/checkpoint_epoch_2600.pt \
  --active \
  --model_base_dim 128 \
  --num_res_blocks 4 \
  --attn_resolutions 16 \
  --num_samples 40000 \
  --batch_size 512 \
  --output_dir results_cifar10_active/fid_samples_2600 \
  --grid_out results_cifar10_active/sample_grid_2600.png \
  --Tp 1e-3 \
  --Ta 6.4 \
  --k 4.0 \
  --tau 0.15

---
## FID Computation

Uses [`clean-fid`](https://github.com/GaParmar/clean-fid) against the CIFAR-10 train set.

In [ ]:
!pip install clean-fid -q

In [ ]:
from cleanfid import fid

FID_KWARGS = dict(
    dataset_name="cifar10",
    dataset_res=32,
    dataset_split="train",
    mode="legacy_pytorch",
    model_name="inception_v3",
)

passive_fid = fid.compute_fid(
    fdir1="results_cifar10_passive/fid_samples_2600",
    **FID_KWARGS,
)

active_fid = fid.compute_fid(
    fdir1="results_cifar10_active/fid_samples_2600",
    **FID_KWARGS,
)

print("=" * 35)
print(f"  Passive FID : {passive_fid:.4f}")
print(f"  Active  FID : {active_fid:.4f}")
print("=" * 35)

with open("fid_results.txt", "w") as f:
    f.write(f"Passive FID: {passive_fid:.6f}\n")
    f.write(f"Active  FID: {active_fid:.6f}\n")